# 02 — Experimentos de RAG: embeddings locais, ChromaDB e recuperação

**Objetivo:** experimentar o pipeline de recuperação de políticas da Empório da Música sem utilizar uma chave de API da OpenAI.

Este notebook utiliza um modelo multilíngue local do Sentence Transformers para gerar embeddings. O conteúdo e a documentação do notebook estão em português, assim como **todas as consultas de recuperação**, alinhadas ao contexto de atendimento da loja.

**Fluxo de trabalho**
1. Carregar o PDF de políticas.
2. Extrair o texto preservando os metadados das páginas.
3. Experimentar tamanhos de fragmentos (*chunks*) e sobreposição.
4. Gerar embeddings multilíngues localmente.
5. Criar e persistir uma coleção no ChromaDB.
6. Executar recuperação semântica com consultas em português.
7. Avaliar a relevância dos resultados recuperados e documentar as decisões de engenharia.

> Modelo: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (embeddings de 384 dimensões). O modelo é baixado no primeiro uso e passa a ser executado localmente. Não é necessária chave da API da OpenAI nem serviço pago de geração de embeddings.

## 0. Instalação das dependências

In [ ]:
%pip -q install -U pypdf sentence-transformers chromadb pandas


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 1. Envio do PDF de políticas

In [ ]:
from pathlib import Path
from google.colab import files
import re
import pandas as pd

print("Selecione o PDF de políticas da loja:")
uploaded = files.upload()

pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
if not pdf_files:
    raise FileNotFoundError("Nenhum arquivo PDF foi enviado.")

PDF_PATH = Path(pdf_files[0])
CHROMA_PATH = Path("/content/chroma_store_policies")
CHROMA_PATH.mkdir(parents=True, exist_ok=True)
COLLECTION_NAME = "store_policies_multilingual_experiment"

print("PDF selecionado:", PDF_PATH.name)
print("Diretório do ChromaDB:", CHROMA_PATH)


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Selecione o PDF de políticas da loja:


Saving políticas_da_loja.pdf to políticas_da_loja.pdf
PDF selecionado: políticas_da_loja.pdf
Diretório do ChromaDB: /content/chroma_store_policies


## 2. Extração do texto do PDF

In [ ]:
from pypdf import PdfReader

reader = PdfReader(str(PDF_PATH))
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    page_text = (page.extract_text() or "").strip()
    if page_text:
        pages.append({
            "text": page_text,
            "page": page_number,
            "source": PDF_PATH.name
        })

if not pages:
    raise ValueError("Não foi possível extrair texto do PDF enviado.")

print(f"Páginas com texto extraído: {len(pages)}")
display(pd.DataFrame([
    {"page": p["page"], "characters": len(p["text"]), "preview": p["text"][:250]}
    for p in pages
]))


Páginas com texto extraído: 8


,page,characters,preview
0,1,397,EMPÓRIO\n \nDA\n \nMÚSICA\n \nSua\n \nmúsica\n...
1,2,2205,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
2,3,1077,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
3,4,2667,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
4,5,1457,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
5,6,2954,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
6,7,2753,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
7,8,1420,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...


## 3. Fragmentação (*chunking*): experimento com tamanho e sobreposição

In [ ]:
def chunk_pages(pages, chunk_size=900, overlap=150):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Utilize chunk_size > overlap >= 0.")

    chunks = []
    for page in pages:
        text = re.sub(r"\\s+", " ", page["text"]).strip()
        start = 0

        while start < len(text):
            end = min(start + chunk_size, len(text))

            if end < len(text):
                boundary = max(
                    text.rfind(". ", start, end),
                    text.rfind("; ", start, end),
                    text.rfind(" ", start, end)
                )
                if boundary > start + int(chunk_size * 0.6):
                    end = boundary + 1

            chunk_text = text[start:end].strip()
            if chunk_text:
                chunks.append({
                    "id": f'{page["source"]}-p{page["page"]}-c{len(chunks) + 1}',
                    "text": chunk_text,
                    "metadata": {
                        "source": page["source"],
                        "page": int(page["page"])
                    }
                })

            if end >= len(text):
                break
            start = max(start + 1, end - overlap)

    return chunks


CHUNK_SIZE = 900
OVERLAP = 150

chunks = chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

chunk_df = pd.DataFrame([
    {
        "id": chunk["id"],
        "page": chunk["metadata"]["page"],
        "characters": len(chunk["text"]),
        "text": chunk["text"]
    }
    for chunk in chunks
])

print("Total de fragmentos:", len(chunks))
display(chunk_df.head(10))


Total de fragmentos: 22


,id,page,characters,text
0,políticas_da_loja.pdf-p1-c1,1,397,EMPÓRIO\n \nDA\n \nMÚSICA\n \nSua\n \nmúsica\n...
1,políticas_da_loja.pdf-p2-c2,2,882,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
2,políticas_da_loja.pdf-p2-c3,2,885,exclusivamente\n \ncom\n \ninstrumentos\n \nmu...
3,políticas_da_loja.pdf-p2-c4,2,732,Telefone\n \n/\n \nWhatsApp\n \n(67)\n \n3341\...
4,políticas_da_loja.pdf-p3-c5,3,897,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
5,políticas_da_loja.pdf-p3-c6,3,328,rcelamento\n \nde\n \n4x\n \na\n \n6x:\n \nsem...
6,políticas_da_loja.pdf-p4-c7,4,886,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
7,políticas_da_loja.pdf-p4-c8,4,894,forma\n \nde\n \npagamento\n \noriginal\n \nem...
8,políticas_da_loja.pdf-p4-c9,4,895,"preferência\n \n(cor,\n \nmodelo,\n \ntamanho)..."
9,políticas_da_loja.pdf-p4-c10,4,431,gião\n \nMetropolitana\n \nde\n \nCampo\n \nGr...


In [ ]:
# Compare diferentes configurações de fragmentação.
chunk_experiments = []

for size, overlap in [(500, 80), (900, 150), (1200, 200)]:
    experiment_chunks = chunk_pages(
        pages,
        chunk_size=size,
        overlap=overlap
    )
    lengths = [len(chunk["text"]) for chunk in experiment_chunks]

    chunk_experiments.append({
        "chunk_size": size,
        "overlap": overlap,
        "number_of_chunks": len(experiment_chunks),
        "average_characters": round(sum(lengths) / len(lengths), 1),
        "minimum_characters": min(lengths),
        "maximum_characters": max(lengths)
    })

display(pd.DataFrame(chunk_experiments))


,chunk_size,overlap,number_of_chunks,average_characters,minimum_characters,maximum_characters
0,500,80,39,444.3,125,498
1,900,150,22,772.2,328,898
2,1200,200,18,939.0,210,1198


## 4. Geração local de embeddings multilíngues (sem chave de API)

O manual de políticas está escrito em português e a loja atende clientes brasileiros. Portanto, um modelo de embeddings multilíngue é mais adequado do que um modelo focado em inglês para consultas em português.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# O modelo é baixado no primeiro uso. Nenhuma chave de API é necessária.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

document_texts = [chunk["text"] for chunk in chunks]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Modelo de embeddings:", EMBEDDING_MODEL_NAME)
print("Quantidade de embeddings:", len(document_embeddings))
print("Dimensões dos embeddings:", document_embeddings.shape)
print("Amostra do primeiro vetor:", document_embeddings[0][:8])


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Quantidade de embeddings: 22
Dimensões dos embeddings: (22, 384)
Amostra do primeiro vetor: [ 0.07008833 -0.03141922 -0.03665372  0.02126042 -0.03228508  0.01241734
 -0.02347486  0.0589514 ]


## 5. Criação e persistência do banco vetorial no ChromaDB

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(CHROMA_PATH))

# Utilize uma coleção separada para os embeddings locais.
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# A operação upsert permite executar novamente esta célula com segurança para IDs de fragmentos idênticos.
collection.upsert(
    ids=[chunk["id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[chunk["metadata"] for chunk in chunks],
    embeddings=document_embeddings.tolist()
)

print("Coleção:", collection.name)
print("Vetores armazenados:", collection.count())
print("Persistido em:", CHROMA_PATH)


Coleção: store_policies_multilingual_experiment
Vetores armazenados: 22
Persistido em: /content/chroma_store_policies


## 6. Recuperação semântica (top-k)

In [ ]:
def retrieve(query, top_k=4):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].tolist()

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(max(1, top_k), collection.count())
    )

    rows = []
    for index, document in enumerate(result["documents"][0]):
        rows.append({
            "rank": index + 1,
            "page": result["metadatas"][0][index].get("page"),
            "distance": result["distances"][0][index],
            "text": document
        })

    return pd.DataFrame(rows)


query = "Qual é o prazo de retorno para uma reclamação?"
display(retrieve(query, top_k=4))


,rank,page,distance,text
0,1,4,0.509212,forma\n \nde\n \npagamento\n \noriginal\n \nem...
1,2,7,0.565042,ossuem\n \ngarantia\n \nlegal\n \nde\n \n90\n ...
2,3,7,0.569170,mar\n \nque\n \no\n \nproduto\n \nnão\n \nfaz\...
3,4,4,0.602710,"preferência\n \n(cor,\n \nmodelo,\n \ntamanho)..."


In [ ]:
# Estas consultas estão intencionalmente em português brasileiro.
# Elas representam perguntas realistas de clientes da loja.
evaluation_queries = [
    "Qual é o prazo de retorno para uma reclamação?",
    "A loja vende acessórios para instrumentos musicais?",
    "O desconto de 5% no PIX pode ser acumulado com uma promoção?",
    "Quais são as condições para troca por preferência?",
    "Quais são os canais de contato da loja?"
]

for query in evaluation_queries:
    print("\\n" + "=" * 100)
    print("CONSULTA:", query)
    display(retrieve(query, top_k=3))


\n====================================================================================================
CONSULTA: Qual é o prazo de retorno para uma reclamação?


,rank,page,distance,text
0,1,4,0.509212,forma\n \nde\n \npagamento\n \noriginal\n \nem...
1,2,7,0.565042,ossuem\n \ngarantia\n \nlegal\n \nde\n \n90\n ...
2,3,7,0.569170,mar\n \nque\n \no\n \nproduto\n \nnão\n \nfaz\...


\n====================================================================================================
CONSULTA: A loja vende acessórios para instrumentos musicais?


,rank,page,distance,text
0,1,2,0.282826,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
1,2,6,0.319219,ões\n \nincorretas\n \ngeram\n \nfrustração\n ...
2,3,2,0.342810,exclusivamente\n \ncom\n \ninstrumentos\n \nmu...


\n====================================================================================================
CONSULTA: O desconto de 5% no PIX pode ser acumulado com uma promoção?


,rank,page,distance,text
0,1,6,0.363107,mbro).\n \n6.2\n \nRegras\n \nde\n \nPromoções...
1,2,7,0.442159,mar\n \nque\n \no\n \nproduto\n \nnão\n \nfaz\...
2,3,3,0.524510,rcelamento\n \nde\n \n4x\n \na\n \n6x:\n \nsem...


\n====================================================================================================
CONSULTA: Quais são as condições para troca por preferência?


,rank,page,distance,text
0,1,4,0.554771,"preferência\n \n(cor,\n \nmodelo,\n \ntamanho)..."
1,2,7,0.573722,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
2,3,3,0.602668,rcelamento\n \nde\n \n4x\n \na\n \n6x:\n \nsem...


\n====================================================================================================
CONSULTA: Quais são os canais de contato da loja?


,rank,page,distance,text
0,1,5,0.530722,em\n \nexigir\n \nfrete\n \nespecial\n \ncom\n...
1,2,6,0.552627,ões\n \nincorretas\n \ngeram\n \nfrustração\n ...
2,3,2,0.603751,Telefone\n \n/\n \nWhatsApp\n \n(67)\n \n3341\...


## 7. Conclusões e próximos passos

A versão 1 permitiu validar o fluxo básico de recuperação semântica utilizando embeddings multilíngues locais e ChromaDB. Foram gerados 22 fragmentos com a configuração de 900 caracteres e sobreposição de 150 caracteres. Os experimentos demonstraram que o pipeline funciona, mas também evidenciaram oportunidades de melhoria na qualidade do texto extraído do PDF e na recuperação de trechos mais diretamente relacionados às consultas. Na versão 2, os principais ajustes serão a normalização e limpeza do texto extraído, reduzindo quebras de linha e separações artificiais entre palavras, e a revisão da estratégia de fragmentação, buscando preservar melhor a estrutura e o contexto das seções do manual. Após essas alterações, os resultados serão comparados utilizando as mesmas consultas, permitindo avaliar se houve melhoria na relevância dos trechos recuperados.